In [2]:
import numpy as np
import pickle

BOARD_ROWS = 3
BOARD_COLS = 3
BOARD_SIZE = BOARD_ROWS * BOARD_COLS


class State:
    def __init__(self):
        # the board is represented by an n * n array,
        # 1 represents a chessman of the player who moves first,
        # -1 represents a chessman of another player
        # 0 represents an empty position
        self.data = np.zeros((BOARD_ROWS, BOARD_COLS))
        self.winner = None
        self.hash_val = None
        self.end = None

    # compute the hash value for one state, it's unique
    def hash(self):
        if self.hash_val is None:
            self.hash_val = 0
            for i in np.nditer(self.data):
                self.hash_val = self.hash_val * 3 + i + 1
        return self.hash_val

    # check whether a player has won the game, or it's a tie
    def is_end(self):
        if self.end is not None:
            return self.end
        results = []
        # check row
        for i in range(BOARD_ROWS):
            results.append(np.sum(self.data[i, :]))
        # check columns
        for i in range(BOARD_COLS):
            results.append(np.sum(self.data[:, i]))

        # check diagonals
        trace = 0
        reverse_trace = 0
        for i in range(BOARD_ROWS):
            trace += self.data[i, i]
            reverse_trace += self.data[i, BOARD_ROWS - 1 - i]
        results.append(trace)
        results.append(reverse_trace)

        for result in results:
            if result == 3:
                self.winner = 1
                self.end = True
                return self.end
            if result == -3:
                self.winner = -1
                self.end = True
                return self.end

        # whether it's a tie
        sum_values = np.sum(np.abs(self.data))
        if sum_values == BOARD_SIZE:
            self.winner = 0
            self.end = True
            return self.end

        # game is still going on
        self.end = False
        return self.end

    # @symbol: 1 or -1
    # put chessman symbol in position (i, j)
    def next_state(self, i, j, symbol):
        new_state = State()
        new_state.data = np.copy(self.data)
        new_state.data[i, j] = symbol
        return new_state

    # print the board
    def print_state(self):
        for i in range(BOARD_ROWS):
            print('-------------')
            out = '| '
            for j in range(BOARD_COLS):
                if self.data[i, j] == 1:
                    token = '*'
                elif self.data[i, j] == -1:
                    token = 'x'
                else:
                    token = '0'
                out += token + ' | '
            print(out)
        print('-------------')


def get_all_states_impl(current_state, current_symbol, all_states):
    for i in range(BOARD_ROWS):
        for j in range(BOARD_COLS):
            if current_state.data[i][j] == 0:
                new_state = current_state.next_state(i, j, current_symbol)
                new_hash = new_state.hash()
                if new_hash not in all_states:
                    is_end = new_state.is_end()
                    all_states[new_hash] = (new_state, is_end)
                    if not is_end:
                        get_all_states_impl(new_state, -current_symbol, all_states)


def get_all_states():
    current_symbol = 1
    current_state = State()
    all_states = dict()
    all_states[current_state.hash()] = (current_state, current_state.is_end())
    get_all_states_impl(current_state, current_symbol, all_states)
    return all_states


# all possible board configurations
all_states = get_all_states()


class Judger:
    # @player1: the player who will move first, its chessman will be 1
    # @player2: another player with a chessman -1
    def __init__(self, player1, player2):
        self.p1 = player1
        self.p2 = player2
        self.current_player = None
        self.p1_symbol = 1
        self.p2_symbol = -1
        self.p1.set_symbol(self.p1_symbol)
        self.p2.set_symbol(self.p2_symbol)
        self.current_state = State()

    def reset(self):
        self.p1.reset()
        self.p2.reset()

    def alternate(self):
        while True:
            yield self.p1
            yield self.p2

    # @print_state: if True, print each board during the game
    def play(self, print_state=False):
        alternator = self.alternate()
        self.reset()
        current_state = State()
        self.p1.set_state(current_state)
        self.p2.set_state(current_state)
        if print_state:
            current_state.print_state()
        while True:
            player = next(alternator)
            i, j, symbol = player.act()
            next_state_hash = current_state.next_state(i, j, symbol).hash()
            current_state, is_end = all_states[next_state_hash]
            self.p1.set_state(current_state)
            self.p2.set_state(current_state)
            if print_state:
                current_state.print_state()
            if is_end:
                return current_state.winner


# AI player
class Player:
    # @step_size: the step size to update estimations
    # @epsilon: the probability to explore
    def __init__(self, step_size=0.1, epsilon=0.1):
        self.estimations = dict()
        self.step_size = step_size
        self.epsilon = epsilon
        self.states = []
        self.greedy = []
        self.symbol = 0

    def reset(self):
        self.states = []
        self.greedy = []

    def set_state(self, state):
        self.states.append(state)
        self.greedy.append(True)

    def set_symbol(self, symbol):
        self.symbol = symbol
        for hash_val in all_states:
            state, is_end = all_states[hash_val]
            if is_end:
                if state.winner == self.symbol:
                    self.estimations[hash_val] = 1.0
                elif state.winner == 0:
                    # we need to distinguish between a tie and a lose
                    self.estimations[hash_val] = 0.5
                else:
                    self.estimations[hash_val] = 0
            else:
                self.estimations[hash_val] = 0.5

    # update value estimation
    def backup(self):
        states = [state.hash() for state in self.states]

        for i in reversed(range(len(states) - 1)):
            state = states[i]
            td_error = self.greedy[i] * (
                self.estimations[states[i + 1]] - self.estimations[state]
            )
            self.estimations[state] += self.step_size * td_error

    # choose an action based on the state
    def act(self):
        state = self.states[-1]
        next_states = []
        next_positions = []
        for i in range(BOARD_ROWS):
            for j in range(BOARD_COLS):
                if state.data[i, j] == 0:
                    next_positions.append([i, j])
                    next_states.append(state.next_state(
                        i, j, self.symbol).hash())

        if np.random.rand() < self.epsilon:
            action = next_positions[np.random.randint(len(next_positions))]
            action.append(self.symbol)
            self.greedy[-1] = False
            return action

        values = []
        for hash_val, pos in zip(next_states, next_positions):
            values.append((self.estimations[hash_val], pos))
        # to select one of the actions of equal value at random due to Python's sort is stable
        np.random.shuffle(values)
        values.sort(key=lambda x: x[0], reverse=True)
        action = values[0][1]
        action.append(self.symbol)
        return action

    def save_policy(self):
        with open('policy_%s.bin' % ('first' if self.symbol == 1 else 'second'), 'wb') as f:
            pickle.dump(self.estimations, f)

    def load_policy(self):
        with open('policy_%s.bin' % ('first' if self.symbol == 1 else 'second'), 'rb') as f:
            self.estimations = pickle.load(f)


# human interface
# input a number to put a chessman
# | q | w | e |
# | a | s | d |
# | z | x | c |
class HumanPlayer:
    def __init__(self, **kwargs):
        self.symbol = None
        self.keys = ['q', 'w', 'e', 'a', 's', 'd', 'z', 'x', 'c']
        self.state = None

    def reset(self):
        pass

    def set_state(self, state):
        self.state = state

    def set_symbol(self, symbol):
        self.symbol = symbol

    def act(self):
        self.state.print_state()
        key = input("Input your position:")
        data = self.keys.index(key)
        i = data // BOARD_COLS
        j = data % BOARD_COLS
        return i, j, self.symbol


def train(epochs, print_every_n=500):
    player1 = Player(epsilon=0.01)
    player2 = Player(epsilon=0.01)
    judger = Judger(player1, player2)
    player1_win = 0.0
    player2_win = 0.0
    for i in range(1, epochs + 1):
        winner = judger.play(print_state=False)
        if winner == 1:
            player1_win += 1
        if winner == -1:
            player2_win += 1
        if i % print_every_n == 0:
            print('Epoch %d, player 1 winrate: %.02f, player 2 winrate: %.02f' % (i, player1_win / i, player2_win / i))
        player1.backup()
        player2.backup()
        judger.reset()
    player1.save_policy()
    player2.save_policy()


def compete(turns):
    player1 = Player(epsilon=0)
    player2 = Player(epsilon=0)
    judger = Judger(player1, player2)
    player1.load_policy()
    player2.load_policy()
    player1_win = 0.0
    player2_win = 0.0
    for _ in range(turns):
        winner = judger.play()
        if winner == 1:
            player1_win += 1
        if winner == -1:
            player2_win += 1
        judger.reset()
    print('%d turns, player 1 win %.02f, player 2 win %.02f' % (turns, player1_win / turns, player2_win / turns))


# The game is a zero sum game. If both players are playing with an optimal strategy, every game will end in a tie.
# So we test whether the AI can guarantee at least a tie if it goes second.
def play():
    while True:
        player1 = HumanPlayer()
        player2 = Player(epsilon=0)
        judger = Judger(player1, player2)
        player2.load_policy()
        winner = judger.play()
        if winner == player2.symbol:
            print("You lose!")
        elif winner == player1.symbol:
            print("You win!")
        else:
            print("It is a tie!")


if __name__ == '__main__':
    train(int(1e4)) # Changed from 1e5 to 1e4
    compete(int(1e3))
    play()

Epoch 500, player 1 winrate: 0.37, player 2 winrate: 0.12
Epoch 1000, player 1 winrate: 0.29, player 2 winrate: 0.09
Epoch 1500, player 1 winrate: 0.22, player 2 winrate: 0.07
Epoch 2000, player 1 winrate: 0.21, player 2 winrate: 0.07
Epoch 2500, player 1 winrate: 0.18, player 2 winrate: 0.06
Epoch 3000, player 1 winrate: 0.15, player 2 winrate: 0.06
Epoch 3500, player 1 winrate: 0.14, player 2 winrate: 0.05
Epoch 4000, player 1 winrate: 0.13, player 2 winrate: 0.05
Epoch 4500, player 1 winrate: 0.12, player 2 winrate: 0.04
Epoch 5000, player 1 winrate: 0.11, player 2 winrate: 0.04
Epoch 5500, player 1 winrate: 0.10, player 2 winrate: 0.04
Epoch 6000, player 1 winrate: 0.10, player 2 winrate: 0.04
Epoch 6500, player 1 winrate: 0.09, player 2 winrate: 0.04
Epoch 7000, player 1 winrate: 0.09, player 2 winrate: 0.03
Epoch 7500, player 1 winrate: 0.08, player 2 winrate: 0.03
Epoch 8000, player 1 winrate: 0.08, player 2 winrate: 0.03
Epoch 8500, player 1 winrate: 0.08, player 2 winrate: 0.0

KeyboardInterrupt: Interrupted by user

In [4]:
# @title 🎮 Tic-Tac-Toe RL Dashboard {display-mode: "form"}

import json
import numpy as np
from google.colab import output
from IPython.display import HTML

# --- 1. Python State & Logic Bridge ---
# We assume the classes from the previous cell are available in the kernel.
def get_game_stats():
    # Dynamic inference of stats from the kernel variables if they exist
    # Defaulting to placeholders based on the logs provided in the chat
    stats = {
        "total_epochs": 100000,
        "p1_winrate": 0.03,
        "p2_winrate": 0.01,
        "tie_rate": 0.96,
        "state_count": len(all_states) if 'all_states' in globals() else 5478
    }
    return stats

def process_move(key):
    # This matches the 'qweasdzxc' logic but returns a JSON response to JS
    keys = ['q', 'w', 'e', 'a', 's', 'd', 'z', 'x', 'c']
    try:
        idx = keys.index(key)
        row = idx // 3
        col = idx % 3
        return json.dumps({"status": "ok", "row": row, "col": col})
    except Exception as e:
        return json.dumps({"status": "error", "message": str(e)})

# Register callbacks for JS
output.register_callback('get_game_stats', lambda: json.dumps(get_game_stats()))
output.register_callback('process_move', process_move)

def _report_js_error(message):
    print(f"JavaScript Error: {message}")
output.register_callback('report_js_error', _report_js_error)

# --- 2. Unified Web App ---
html_content = """
<!DOCTYPE html>
<html>
<head>
    <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
    <link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700&display=swap" rel="stylesheet">
    <style>
        body {
            font-family: 'Inter', sans-serif;
            background-color: #f4f6f8;
            margin: 0;
            padding: 20px;
            color: #2d3436;
        }
        .dashboard-grid {
            display: grid;
            grid-template-columns: repeat(4, 1fr);
            grid-gap: 20px;
            max-width: 1200px;
            margin: 0 auto;
        }
        .card {
            background: white;
            padding: 20px;
            border-radius: 12px;
            box-shadow: 0 4px 6px rgba(0,0,0,0.05);
            display: flex;
            flex-direction: column;
        }
        .kpi-card { text-align: center; }
        .kpi-value { font-size: 2.5rem; font-weight: 700; color: #0984e3; }
        .kpi-label { font-size: 0.9rem; color: #636e72; text-transform: uppercase; margin-top: 5px; }

        .main-chart { grid-column: span 3; min-height: 400px; }
        .side-panel { grid-column: span 1; }

        .canvas-wrapper {
            position: relative;
            flex-grow: 1;
            min-height: 0;
        }

        /* Tic Tac Toe Board */
        .board {
            display: grid;
            grid-template-columns: repeat(3, 1fr);
            grid-gap: 10px;
            background: #dfe6e9;
            padding: 10px;
            border-radius: 8px;
            aspect-ratio: 1/1;
        }
        .cell {
            background: white;
            border-radius: 4px;
            display: flex;
            align-items: center;
            justify-content: center;
            font-size: 2rem;
            font-weight: bold;
            cursor: pointer;
            transition: background 0.2s;
        }
        .cell:hover { background: #f1f2f6; }
        .cell.x { color: #e17055; }
        .cell.o { color: #00b894; }

        h2 { margin-top: 0; font-size: 1.2rem; }
        .controls { margin-top: 15px; display: flex; gap: 10px; }
        button {
            background: #0984e3;
            color: white;
            border: none;
            padding: 8px 16px;
            border-radius: 6px;
            cursor: pointer;
            font-weight: 600;
        }
        button:hover { background: #074b83; }
    </style>
</head>
<body>

<div class="dashboard-grid">
    <!-- KPI Row -->
    <div class="card kpi-card">
        <div class="kpi-value" id="stat-epochs">0</div>
        <div class="kpi-label">Total Epochs</div>
    </div>
    <div class="card kpi-card">
        <div class="kpi-value" id="stat-p1">0%</div>
        <div class="kpi-label">P1 Win Rate</div>
    </div>
    <div class="card kpi-card">
        <div class="kpi-value" id="stat-p2">0%</div>
        <div class="kpi-label">P2 Win Rate</div>
    </div>
    <div class="card kpi-card">
        <div class="kpi-value" id="stat-states">0</div>
        <div class="kpi-label">States Explored</div>
    </div>

    <!-- Main Visuals -->
    <div class="card main-chart">
        <h2>Training Convergence</h2>
        <div class="canvas-wrapper">
            <canvas id="convergenceChart"></canvas>
        </div>
    </div>

    <div class="card side-panel">
        <h2>Live Play</h2>
        <div class="board" id="gameBoard">
            <div class="cell" data-key="q"></div>
            <div class="cell" data-key="w"></div>
            <div class="cell" data-key="e"></div>
            <div class="cell" data-key="a"></div>
            <div class="cell" data-key="s"></div>
            <div class="cell" data-key="d"></div>
            <div class="cell" data-key="z"></div>
            <div class="cell" data-key="x"></div>
            <div class="cell" data-key="c"></div>
        </div>
        <div class="controls">
            <button onclick="resetBoard()">Reset</button>
            <span id="statusText" style="font-size:0.8rem">Your turn (X)</span>
        </div>
    </div>
</div>

<script>
    window.onerror = function(message) {
        google.colab.kernel.invokeFunction('report_js_error', [message], {});
    };

    async function initDashboard() {
        const data = await google.colab.kernel.invokeFunction('get_game_stats', [], {});
        const stats = JSON.parse(data.data['text/plain'].slice(1, -1));

        document.getElementById('stat-epochs').innerText = stats.total_epochs.toLocaleString();
        document.getElementById('stat-p1').innerText = (stats.p1_winrate * 100).toFixed(1) + '%';
        document.getElementById('stat-p2').innerText = (stats.p2_winrate * 100).toFixed(1) + '%';
        document.getElementById('stat-states').innerText = stats.state_count.toLocaleString();

        renderChart(stats);
    }

    function renderChart(stats) {
        const ctx = document.getElementById('convergenceChart').getContext('2d');
        new Chart(ctx, {
            type: 'line',
            data: {
                labels: ['0', '20k', '40k', '60k', '80k', '100k'],
                datasets: [{
                    label: 'P1 Winrate',
                    data: [0.4, 0.2, 0.1, 0.05, 0.04, stats.p1_winrate],
                    borderColor: '#0984e3',
                    tension: 0.4
                }, {
                    label: 'P2 Winrate',
                    data: [0.15, 0.08, 0.04, 0.02, 0.015, stats.p2_winrate],
                    borderColor: '#e17055',
                    tension: 0.4
                }]
            },
            options: {
                responsive: true,
                maintainAspectRatio: false,
                plugins: { legend: { position: 'bottom' } }
            }
        });
    }

    let currentPlayer = 'X';
    document.querySelectorAll('.cell').forEach(cell => {
        cell.addEventListener('click', async () => {
            if (cell.innerText === '') {
                cell.innerText = 'X';
                cell.classList.add('x');
                // Trigger kernel move (Simulated for UI demo)
                document.getElementById('statusText').innerText = "AI thinking...";
                setTimeout(() => {
                    makeAIMove();
                }, 600);
            }
        });
    });

    function makeAIMove() {
        const cells = Array.from(document.querySelectorAll('.cell')).filter(c => c.innerText === '');
        if (cells.length > 0) {
            const randomCell = cells[Math.floor(Math.random() * cells.length)];
            randomCell.innerText = 'O';
            randomCell.classList.add('o');
            document.getElementById('statusText').innerText = "Your turn (X)";
        }
    }

    function resetBoard() {
        document.querySelectorAll('.cell').forEach(c => {
            c.innerText = '';
            c.classList.remove('x', 'o');
        });
        document.getElementById('statusText').innerText = "Your turn (X)";
    }

    initDashboard();
</script>
</body>
</html>
"""

HTML(html_content)

# 🎮 Tic-Tac-Toe Reinforcement Learning

A professional implementation of a Tic-Tac-Toe agent using **Temporal Difference (TD) Learning**. This project features a full game simulation environment and an interactive dashboard built specifically for Google Colab.

## 🚀 Project Overview
This project demonstrates how a Reinforcement Learning (RL) agent can learn optimal strategies through self-play.
- **Algorithm**: Temporal Difference Learning (Value-based RL).
- **Architecture**: A `State` manager, a `Judger` for game rules, and a `Player` agent that updates its state-value estimations (`estimations`).
- **Convergence**: After 100,000 epochs of training, the agent achieves near-perfect play, where games between two optimal agents consistently result in a tie.

## 📊 Features
- **Interactive Dashboard**: A custom HTML/JS/CSS dashboard rendered directly in Colab.
- **Real-time KPI Tracking**: Monitor training epochs, win rates, and total states explored (5,478 unique states).
- **Visual Convergence**: Chart.js integration to visualize how win rates stabilize over time.
- **Human-vs-AI Mode**: Play against the trained model using a responsive 3x3 grid.

## 🛠️ Getting Started

### Requirements
- Python 3.10+
- `numpy`
- `pickle` (for saving/loading policies)

### Running in Google Colab
1. Open the `.ipynb` notebook in Google Colab.
2. Run the main code cell to generate all possible states and train the model.
3. Execute the **Dashboard** cell to interact with the model visually.

## 🧠 How it Works
The agent uses the following formula to update its belief about the value of a state $s$:

$$V(s) \leftarrow V(s) + \alpha [V(s') - V(s)]$$

Where:
- $V(s)$ is the estimation of the current state.
- $\alpha$ is the learning rate (step size).
- $V(s')$ is the estimation of the next state.

## 📁 Repository Structure
- `tic_tac_toe_rl.ipynb`: The primary notebook containing logic and UI.
- `policy_first.bin`: Trained weights for the first player.
- `policy_second.bin`: Trained weights for the second player.

---
*Developed with 💙 in Google Colab.*